In [0]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("Silver_Validation").getOrCreate()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
Rules_Data = [
    ("MEMBER_PERSON_ID_NOT_NULL",["person_id"]),
    ("MEMBER_APPLICATION_ID_NOT_NULL",["appln_id"]),
    ("MEMBER_NAME_REQUIRED",["first_name","last_name"]),
    ("MEMBER_DOB_VALID",["dbo"]),
    ("MEMBER_GENDER_CODE_VALID",["gender_cd"]),
    ("MEMBER_STATUS_CODE_VALID",["person_status_cd"]),
    ("MEMBER_RECORD_HASH_REQUIRED",["record_hash"]),
    ("MEMBER_RUN_ID_REQUIRED",["run_id"]),
    ("MEMBER_SOURCE_EVENT_TS_REQUIRED",["source_event_ts"]),
    ("MEMBER_CREATED_TS_REQUIRED",["created_ts"]),
    ("MEMBER_SUBSCRIBER_REQUIRED_FOR_DEPENDENT",["is_dependent_ind","subscriber_person_id"]),
    ("MEMBER_RELATIONSHIP_CODE_VALID",["relationship_cd"]),
    ("MEMBER_HISTORY_NO_OVERLAP",["person_id","effective_start_ts","effective_end_ts"]),
    ("MEMBER_ONE_CURRENT_ROW",["person_id","is_current_flag"]),

]

In [0]:
Rules_Data_Test = [
    ("MEMBER_PERSON_ID_NOT_NULL",["person_id"]),
    ("MEMBER_APPLICATION_ID_NOT_NULL",["appln_id"]),
    ("MEMBER_NAME_REQUIRED",["first_name","last_name"])
]

In [0]:
Rules_DF_Test = spark.createDataFrame(Rules_Data_Test,["Rule","Columns"])
display(Rules_DF_Test)

In [0]:
Rules_DF = spark.createDataFrame(Rules_Data,["Rule","Columns"])
display(Rules_DF)

In [0]:
Rules_DF_Explode = Rules_DF.withColumn("Fields_Involved",explode("Columns")).drop(col("Columns"))
display(Rules_DF_Explode)

In [0]:
display(Rules_DF.printSchema())

In [0]:
%sql 
CREATE TABLE workspace.bronze.member (
  person_id BIGINT,
  appln_id BIGINT,
  first_name STRING,
  last_name STRING,
  dob DATE,
  gender_cd STRING,
  person_status_cd STRING,
  record_hash STRING,
  run_id STRING,
  source_event_ts TIMESTAMP,
  created_ts TIMESTAMP,
  is_dependent_ind BOOLEAN,
  subscriber_person_id BIGINT,
  relationship_cd STRING,
  effective_start_ts TIMESTAMP,
  effective_end_ts TIMESTAMP,
  is_current_flag BOOLEAN
)
USING DELTA;

In [0]:
%sql
SELECT * FROM workspace.bronze.member ORDER BY person_id;

In [0]:
%sql
INSERT INTO workspace.bronze.member VALUES
(NULL, 5001, 'John', 'Smith', '1985-05-20', 'M', 'A', 'abc123def456', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1001, 'SELF', '2026-01-01 00:00:00', '9999-12-31 23:59:59', true),
(1002, NULL, 'Jane', 'Smith', '1988-08-15', 'F', 'A', 'ghi789jkl012', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', true, 1001, 'SPOUSE', '2026-01-01 00:00:00', '9999-12-31 23:59:59', true),
(1003, 5001, NULL, 'Smith', '2015-02-10', 'M', 'A', 'mno345pqr678', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', true, 1001, 'CHILD', '2026-01-01 00:00:00', '9999-12-31 23:59:59', true)

In [0]:
%sql
INSERT INTO workspace.bronze.member VALUES
(1004, 5002, 'Emily', 'Jones', '1992', 'F', 'A', 'stu901vwx234', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1004, 'SELF', '2025-11-15 00:00:00', '9999-12-31 23:59:59', true),
(1005, 5003, 'Michael', 'Brown', '1978-03-25', 'NA', 'I', 'yz567abc890', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1005, 'SELF', '2020-07-20 00:00:00', '2024-12-31 23:59:59', false),
(1006, 5004, 'Sarah', 'Davis', '2001-07-07', 'F', 'I', 'def123ghi456', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1006, 'SELF', '2023-08-01 00:00:00', '9999-12-31 23:59:59', true)

In [0]:
%sql
INSERT INTO workspace.bronze.member VALUES
(1011, 5008, 'Chris', 'Taylor', '1982-10-22', 'M', 'I', 'ghi123jkl456', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1011, 'SELF', '2019-01-15 00:00:00', '2023-10-31 23:59:59', false),
(1012, 5009, 'Amanda', 'Anderson', '1994-04-14', 'F', 'A', 'mno789pqr012', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1012, 'SELF', '2021-03-01 00:00:00', '9999-12-31 23:59:59', true),
(1013, 5010, 'James', 'Thomas', '1976-01-09', 'M', 'A', 'stu345vwx678', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1013, 'SELF', '2018-11-11 00:00:00', '9999-12-31 23:59:59', true),
(1014, 5010, 'Susan', 'Thomas', '1977-05-19', 'F', 'A', 'yz901abc234', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', true, 1013, 'SPOUSE', '2018-11-11 00:00:00', '9999-12-31 23:59:59', true),
(1015, 5011, 'Daniel', 'Jackson', '2005-08-29', 'M', 'A', 'def567ghi890', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1015, 'SELF', '2023-09-01 00:00:00', '9999-12-31 23:59:59', true),
(1016, 5012, 'Laura', 'White', '1998-02-28', 'F', 'A', 'jkl123mno456', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1016, 'SELF', '2024-06-15 00:00:00', '9999-12-31 23:59:59', true)

In [0]:
%sql
INSERT INTO workspace.bronze.member VALUES
(1007, 5005, 'David', 'Miller', '1995-09-12', 'M', 'A', NULL, 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1007, 'SELF', '2022-02-18 00:00:00', '9999-12-31 23:59:59', true),
(1008, 5005, 'Olivia', 'Miller', '2021-04-05', 'F', 'A', 'pqr345stu678', NULL, '2026-04-09 22:53:51', '2026-04-09 22:53:51', true, 1007, 'CHILD', '2022-02-18 00:00:00', '9999-12-31 23:59:59', true),
(1009, 5006, 'Robert', 'Wilson', '1989-12-01', 'M', 'A', 'vwx901yz234', 'run-20260409-1', NULL, '2026-04-09 22:53:51', false, 1009, 'SELF', '2024-05-10 00:00:00', '9999-12-31 23:59:59', true),
(1010, 5007, 'Jessica', 'Moore', '1999-06-18', 'F', 'A', 'abc567def890', 'run-20260409-1', '2026-04-09 22:53:51', NULL, false, 1010, 'SELF', '2025-01-20 00:00:00', '9999-12-31 23:59:59', true)

In [0]:
%sql
INSERT INTO workspace.bronze.member VALUES
(1017, 5013, 'Kevin', 'Harris', '1984-07-17', 'M', 'A', 'pqr789stu012', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', true, NULL, 'SELF', '2020-01-01 00:00:00', '9999-12-31 23:59:59', true),
(1018, 5014, 'Nancy', 'Martin', '1991-03-03', 'F', 'I', 'vwx345yz678', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1018, 'ME', '2019-05-20 00:00:00', '2022-05-19 23:59:59', true),
(1019, 5015, 'Paul', 'Thompson', '1965-12-12', 'M', 'A', 'abc901def234', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1019, 'SELF', '2015-02-01 00:00:00', '9999-12-31 23:59:59', true),
(1019, 5016, 'Donna', 'Garcia', '1996-10-16', 'F', 'A', 'ghi567jkl890', 'run-20260409-1', '2026-04-09 22:53:51', '2026-04-09 22:53:51', false, 1020, 'SELF', '2024-03-10 00:00:00', '9999-12-31 23:59:59', true);

In [0]:
def get_rule_condition(rule_name ,cols):
    if rule_name == "MEMBER_PERSON_ID_NOT_NULL":
        return col(cols[0]).isNull()
    elif rule_name == "MEMBER_APPLICATION_ID_NOT_NULL":
        return col(cols[0]).isNull()
    elif rule_name == "MEMBER_NAME_REQUIRED":
        return col(cols[0]).isNull() | col(cols[1]).isNull()
    return None

In [0]:
df_bronze = spark.read.table("workspace.bronze.member")


In [0]:
%sql
SELECT * FROM workspace.bronze.member ORDER BY person_id

In [0]:
bad_df1 = df_bronze.limit(0)
print(bad_df)

In [0]:
for row in Rules_DF_Test.collect():
    rule_name = row['Rule']
    cols = row['Columns']
    condition = get_rule_condition(rule_name,cols)
    failed_df = df_bronze.filter(condition)
    if failed_df is None:
        bad_df1 = failed_df
    else:
        bad_df1 = bad_df1.unionByName(failed_df)

In [0]:
display(bad_df1)